#### 1. PyTorch Setup

In [ ]:
import torch

print(torch.__version__) # PyTorch Version
print(torch.mps.is_available()) # For Mac OS - True
print(torch.backends.mps.is_available()) # For Mac OS - True
print(torch.mps.device_count()) # No of devices, - Only 1
print("mps" if torch.mps.is_available() else "cpu")

# ========== MPS / CUDA & GPU ================
# 1. Only 1 device, MPS backend treats entire GPU as single device
# 2. No Multi-GPU Support, CUDA's alternative is MPS
# 3. True Multiple GPUS is possible on system with discrete GPUs (NVIDIA cards)
# 4. To check cores - About this Mac > More info > Hardware ?
# 4.1 Run on terminal `system_profiler SPDisplayDataType`

# Mac has an Apple Silicon chip, to accelerate PyTorch code.

2.7.0
True
True
1
mps


#### 2. CUDA vs MPS, in more depth

In [22]:
if torch.mps.is_available():
    device = torch.device("mps")
    x = torch.randn(3, 3).to(device)
    print("Running on MPS:", x.device)
else:
    print("MPS not available")

if torch.cuda.is_available():
    device = torch.device("cuda")
    x = torch.randn(3, 3).to(device)
    print("Running on CUDA:", x.device)
else:
    print("CUDA not available")

Running on MPS: mps:0
CUDA not available


#### 3. Common Tensor Operations

In [ ]:
# 1. Tensors are data-containers for array-like
tensor0d = torch.tensor(1) # 0d Tensor
tensor1d = torch.tensor([1, 2, 3]) # 1D Tensor
tensor2d = torch.tensor([[1, 2, 3], [3, 4, 4]]) # 2D Tensor, From Nested Python List
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[1, 6], [2, 9]]])

# 2. Tensor DataTypes
print(tensor1d.dtype)

# Float Data Type
floatvec = torch.tensor([1.0, 3.0, 5.0, 6.9])
print(floatvec.dtype) # 32-bit 
# ===============
# A 32 bit offers sufficient precision, consume less memory & resources
# Most GPU Arch are optimized for 32-Bits computations.
# Hence Speed Up Model Trainig & inference
# ===============

# Possible to change the precision using `.to` method
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

# 3 Tensor Operations
print(tensor2d)
print(tensor2d.shape) # Tensor has 2 rows and 3 columns
print(tensor2d.reshape(3, 2))

print(tensor3d.reshape(4, -1).shape) 
# Use of -1 to let PyTorch infer correct dimension

# Check `.is_contiguous()` for checking memory allocation
# Use .contiguous() to get a contiguous copy if needed 
# (e.g., before using .view() on a non-contiguous tensor).
tensor2d.view(3, 2) # Most common way to reshape
tensor2d.T # Transpose the tensor, flipping it across its diagonal
print(tensor2d.transpose(0, 1)) # Swaps dim 0 and dim 1
print(tensor2d.T.is_contiguous()) # False - Changes in logical view

# Matrix Multiplication
tensor2d.matmul(tensor2d.T) # matmul or @
tensor2d @ tensor2d.T

torch.int64
torch.float32
torch.float32
tensor([[1, 2, 3],
        [3, 4, 4]])
torch.Size([2, 3])
tensor([[1, 2],
        [3, 3],
        [4, 4]])
torch.Size([4, 2])
tensor([[1, 3],
        [2, 4],
        [3, 4]])
False


tensor([[14, 23],
        [23, 41]])

#### 4. Tensor Manipulation

##### Why do we manipulate Tensor Dimensions ?

1. Batching - Models process multiple samples at once (batch dimension)
2. Layer Requirement - Expect inputs in certain shapes
3. Multi-head Attention - Require Splitting & merging dimensions for heads.
4. Broadcasting - Operations like addition/multiplication may require matching shapes

In [ ]:
x = torch.randn(2, 3, 4)  # create a random tensor with shape (2, 3, 4, 5)
# shape (2, 3, 4, 5) means:
# 2 matrices, each with 3 rows and 4 columns, and 
# each element is a vector of size 5
print(x.shape)  # print the tensor
y = x.transpose(0, 1) # transpose the first two dimensions
# Switching betweeen the [batch, seuence, feature] and [sequence, batch, feature] formats
print(y.shape)  # print the transposed tensor

# Again back to [batch, sequence, feature]
y = y.permute(1, 0, 2)
print(y.shape)

torch.Size([2, 3, 4])
torch.Size([3, 2, 4])
torch.Size([2, 3, 4])


In [ ]:
# Reshaping / view
# Changes shape of tensor without changing its data.
# Used to flatten images, prepare batches,

x = torch.arange(6) # Shape [6]
x_reshaped = x.view(2, 3) # Shape [2, 3]

# Use -1 to let library infer correct dimension. x.view(-1, 3)

# 1. Flattening for Fully Connected Layers
# linear layer expects [batch, features], not [batch, channesl, height, width]
x = torch.randn(32, 3, 28, 28) # [batch, channesl, height, width]
x_flat = x.view(32, -1) # [batch, features]

# 2. Adding a Batch Dimension
# if you have single sample but model expect a batch
x = torch.randn(10) # [features]
x_batch = x.unsqueeze(0) # [1, features]

# 3. Preparing Sequnces for RNNs
# PyTorch RNNs expect [seq, batch, features]
x = torch.randn(64, 10, 128) # [batch, seq, features]
x_seq_first = x.permute(1, 0, 2) # [seq, batch, features]

# 4. unsqueeze - adds a new dim of size 1 at specified position (axis)
# enables broadcasting, expected input shape of a layer
x = torch.tensor([1, 2, 3]) # Shape [3]
x1 = x.unsqueeze(0) # shape: [1, 3]
x2 = x.unsqueeze(1) # shape: [3, 1]

# Suppose model expects [batch, features], but you have a single feature

# 5. squeeze - removes all dim of size
# reduce rank of tensor oprations
y = torch.rann(1, 3, 1, 5)
y1 = y.squeeze() # Shape: [3, 5], (removes all size-1 dimes)
y2 = y.squeeze(2) # Shape: [1, 3, 5] (removes only dim 2)

In [ ]:
# 6. cat - joins a squence of tensor along an existing dims
a = torch.randn(2, 3)
b = torch.randn(2, 3)

cat0 = torch.cat([a, b], dim=0) # concatenate along rows (dim=0): shape [4, 3]
cate1 = torch.cat([a, b], dim=1) # along columns (dim=1): shape [2, 6]

# 7. stack - Squence of tensor along a new dim
stack0 = torch.stack([a, b], dim=0) # row, result shape [2, 2, 3]
stack1 = torch.stack([a, b], dim=1) # column, shape[2, 2, 3]

In [ ]:
# 8. split - divide a tensor into a list of smaller tensor of specified size(s)
# useful for dividing data into mini-bath, splitting features
x = torch.arnage(12).reshape(3, 4) # Shape [3, 4]
splits = torch.split(x, 2, dim=1) # column, split into 2. # [3, 2] [3, 2]

# 9. chunnk - divide into specified number of equal chunks
chunks = torch.chunk(x, 2, dim=0) # 2 parts along rows. #[2, 4] [1, 4]

In [ ]:
# 10. Reduction Operations (sum, mean, max, min)
# collapse one or more dims. Specify the dim to reduce
x = torch.tensor([[1., 2.], [3., 4.]])
total = x.sum()
row_sum = x.sum(dim=0) # [4., 6.]
col_mean = x.mean(dim=1) #[1.5, 3.5]
max_val, max_idx = x.max(dim=1) #([2., 4.], [1, 1]) # max val & its index

# loss calculation, loss = (pred - target).pow(2).mean()
# pooling layers: torch.maxpool2d
# Normalization: x - x.mean(dim=0)

In [14]:
# 11. Matrix Multiplications, Dot Prodction
# Use @ or torch.matmul for matrix multiplication
# Use torch.dot for 1D vectors
# Linear Layer = x @ W.T + b
# Attention Score = Q @ K.T

a = torch.randn(2, 3)
b = torch.randn(3, 4)

# Matrix Multiplication: [2, 3] @ [3, 4] -> [2, 4]
c = a @ b

# Dot Prodcut : [3], [3] -> Scalar
v1= torch.tensor([1., 2., 3.])
v2 = torch.tensor([4., 5., 6.])
dot = torch.dot(v1, v2)


#### 5. Seeing Model as Computational Graph

In [37]:
# PyTorch’s automatic differentiation engine, also known as autograd
# to compute gradients in dynamic computational graphs automatically

# computation graph  -> lays out the sequence of calculations needed to 
# compute the output of a neural network – would be required 
# to compute the required gradients for backpropagation, 
# which is the main training algorithm for neural networks.
import torch.nn.functional as F

y = torch.tensor([1.0]) # True label
x1 = torch.tensor([1.1]) # Input Feature
w1 = torch.tensor([2.2]) # weight parameter
b1 = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b1 # net input
a = torch.sigmoid(z) # sigmoid activation

loss = F.binary_cross_entropy(a, y) # predicted probab vs true label
# BCE(a, y) = - [y.log(a) + (1-y).log(1-a)]
# P(y|a) = a^y.(1-a)^(1-y), Bernoulli's PMF
print(loss)

# Can use gradient of loss function w.r.t w1 & b1 (model parameters), 
# to train model


tensor(0.0852)


In [ ]:
# What is sigmoid activation ?
# - 1 / ( 1 + e^(-x)). Real-value to range of 0 to 1. 
# large positive approaches 1, large negative approaches 0. 

# Is it symmetry ? 
# The sigmoid function is not symmetric about the y-axis (not an even function),
# but it is symmetric about the point (0, 0.5)

x = torch.tensor([-50.0])
y = torch.sigmoid(x) # Output - 0
print(y)

tensor([1.9287e-22])


#### 6. Automatic Differentiation Made Easy

In [23]:
# build such a graph internally by default if one of its terminal nodes 
# has the requires_grad attribute set to True.

# Gradients are required when training neural networks
# via the popular backpropagation algorithm

# Partial Derivative - Rate at which function changes w.r.t to one of its variables
# Gradient - Vector of Partial Derivative of mutivariate function
# Provides info to update each of the parameter that minimizes loss function (gradient descent)
# loss function serves as proxy for the model performance, 

# PyTorch’s autograd engine constructs a computational graph 
# in the background. Then, calling the grad function, 
# we can compute the gradient of the loss with respect to model parameter w1 

import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b1 = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b1
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b1 = grad(loss, b1, retain_graph=True)

# PyTorch destroys the computation graph after calculating the gradients 
# to free memory, hence `retain_graph=True`

print(grad_L_w1)
print(grad_L_b1)

# we can call .backward on the loss, and PyTorch will compute 
# the gradients of all the leaf nodes in the graph, 
# which will be stored via the tensors’ .grad attributes:

loss.backward()
print(w1.grad)
print(b1.grad)

(tensor([-0.0898]),)
(tensor([-0.0817]),)
tensor([-0.0898])
tensor([-0.0817])


#### 7. PyTorch Parameters (`.parameters()`)

In [10]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)

# PyTorch makes two tensors for the weights and biases.
# Special because Pytorch marks them as things it should changes during Training.
# When we call model.parameters(), it returns these tensors.
model = MyModel()
for param in model.parameters():
    print(param.shape)
    print(param)

torch.Size([2, 4])
Parameter containing:
tensor([[-0.2013,  0.1039,  0.1993,  0.4580],
        [ 0.1079, -0.2671,  0.1110, -0.2548]], requires_grad=True)
torch.Size([2])
Parameter containing:
tensor([-0.0651, -0.4072], requires_grad=True)


In [ ]:
import torch
import torch.nn as nn

w = nn.Parameter(torch.randn(2, 2))
print(isinstance(w, nn.Parameter))

# nn.Parameter is a special kind of tensor that is automatically registered as a parameter in the module.
# It is used to define learnable parameters in a neural network.
# nn.Parameter is a subclass of torch.Tensor, so it behaves like a tensor.

# If you add this to a module, it will show up in .parameters()
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.my_weight = nn.Parameter(torch.randn(2, 2))

model = MyModel()
print(list(model.parameters()))


#### 8. Implementing Multilayer Neural Network

In [ ]:
import torch
class NeuralNetwork(torch.nn.Module):
    # torch.nn.Module has a __call__ method. 
    # gets invoked when nn.Module instance - model(X)
    # __call__ method is responsible for calling `forward` method
    # 1. Hooks - pre-forward & post-forward hooks, fn to register executed before and after 
    # 2. Parameter Checks, # 3. Automatic Differntiation Setup # 4. Device Mgmt, if already inp & output moved

    def __init__(self, num_in, num_out):
        super().__init__()

        self.layers = torch.nn.Sequential(

            torch.nn.Linear(num_in, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            torch.nn.Linear(20, num_out),
            # Output of last layer
            # No passing to a nonlinear activation fn.
            # combine the softmax operation with negative log-likelihood loss in a single class
            # due to numerical efficiency and stability


        )
    def forward(self, x):
        logits = self.layers(x) # As Sequential is already part of __init__
        return logits

#=========Model Arch ============#
model = NeuralNetwork(50, 3)
print(model)

#============ Parameters =========#
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainable model parameters : ", num_params)
# each parameter which requires_grad=True, count as trainable parameter
# these are contained in nn.Linear layers (Fully Connected Layer)
print(model.layers[0].weight.shape)
# model weights are initalized with small random numbers - to break symmetry during training

#========== Model Call ============#
torch.manual_seed(123)
X = torch.rand((1, 50)) # a single random training example with 50 features
out = model(X) # it automatically executes the forward pass of the model ?
print(out)

# Returns three scores, and grad_fn - Which is used by PyTorch to compute gradients
# If we just use for prediction after training, constructing CP for backpropogation can be wasteful
# unnecessary computations and consumes additional memory
# Hence use torch.no_grad() context manager, does not keep track of gradient

#========== Inference =============#
with torch.no_grad():
#    out = model(X)
    out = torch.softmax(model(X), dim=1)
print(out)
# The values can be interpreted as class-membership that sum up to 1

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
Total number of trainable model parameters :  2213
torch.Size([30, 50])
tensor([[-0.0879,  0.1729,  0.1534]], grad_fn=<AddmmBackward0>)
tensor([[0.2801, 0.3635, 0.3565]])


#### 9. Self-Attention Layer (Transformers)

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

tokens = ["The", " ", "cat", " ", "sat", " ", "on", " ", "the", " ", "mat", "."]
n_tokens = len(tokens)
d_k = 6

# randomly initialize Q, K, V with Standard Normal distribution (mean=0, std=1)
Q = torch.randn(n_tokens, d_k) # n_tokens x d_k
K = torch.randn(n_tokens, d_k)
V = torch.randn(n_tokens, d_k)

# (n_tokens x d_k) @ (d_k x n_tokens) = (n_tokens x n_tokens)
scores = Q @ K.T 

# Values can become large, so we scale them down by the square root of d_k
# to prevent softmax from saturating
# scaling keeps variance of the dot product more consistent
# (n_tokens x n_tokens) / sqrt(d_k) = (n_tokens x n_tokens)
scaled_score = scores / (d_k ** 0.5)

# softmax to get attention weights last dimension
# For each query, softamx is applied across all keys
# converts each row to probaility distribution
# the last diimension corresponds to the keys
attn_weights = F.softmax(scaled_score, dim=-1)

# (n_tokens x n_tokens) @ (n_tokens x d_k) = (n_tokens x d_k)
# the attention weights are used to weight the values
# the result is a weighted sum of the values
output_original = attn_weights @ V

output_original

tensor([[ 0.3341, -0.5154, -1.2380, -0.2892, -0.4579, -0.2457],
        [-0.6077, -0.0793,  1.2263,  0.4887, -0.1040, -0.6966],
        [-0.2376,  0.6978, -0.2318, -0.5215,  0.0550, -0.2912],
        [-0.1975,  0.7894, -0.4018, -0.5038,  0.0247, -0.5158],
        [ 0.0433,  0.2650, -0.5138, -0.3914, -0.2075, -0.0951],
        [-0.3837,  0.6075,  0.4693, -0.1916, -0.1801, -0.0152],
        [ 0.2334, -0.6505, -1.1035, -0.1337, -0.4387, -0.4416],
        [-0.1326,  0.3576, -0.4958, -0.5872, -0.0889, -0.1419],
        [ 0.3356, -0.6253, -1.3418, -0.2246, -0.4712, -0.3358],
        [-0.1198,  0.5244, -0.3412, -0.5332, -0.0867, -0.0930],
        [-0.0488,  0.1532, -0.6066, -0.5057, -0.1759, -0.2830],
        [-0.1535,  0.2497, -0.4927, -0.4406, -0.1209, -0.2047]])

#### 10. Setting up efficient data loaders

In [ ]:
# Dataset class is used to define how each record is loaded
# DataLoader handles how the data is shuffled and assembled into batches
from torch.utils.data import Dataset, DataLoader

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

# Class Label Numbering
# - Class labels to start with 0 and largest class label value should not exceed
# - number of output nodes minus 1

# use to instantiate DataLoader
class ToyDataset(Dataset):
    def __init__(self, X, y):
        # setup attributes that can be accessed in __getitem__ or __len__
        # as we created tensor X, y that sits in memoty, simply assigning
        # to our placeholder objects
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        # returning exactly one item from the dataset via index
        one_x = self.features[index]
        one_y = self.labels[index]

        return one_x, one_y

    
    def __len__(self):
        # lenght of dataset
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0, # crucial for parallelizing data laoding & preprocessing
    # as set to 0, data loading done in main process
    # CPU will take time to load & preprocess the data. As result GPU is idle
    # num worker set to greater than - are launched to load data in parallel
    # freeing main process to focus on training model
    
    # On Jupyter Notebook = increasing num_Workers may not provide noticeable speedup
    # potential issues of overhead spinning up multiple workers, hence longer
    # sharing resources between different process, resulting in errors / nb crashes

    # setting num_works=4 leads to optimal performance
    #drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

for idx, (x, y) in enumerate(train_loader):
    # train_loader iterates over train_ds, each example once = training epoch
    print(f"batch {idx+1}:", x, y)

# since 3rd batch only contains one single example. Smaller batch as last batch
# can disturb the convergence during training. therefore drop_last=True

batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


#### 11. A Typical Training loop

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_in=2, num_out=2) # 2 input feature , 2 class label
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# Validation dataset is similar to test set,
# While we only want to use a test set precisely once to avoid
# biasing the evaluation, we use valid set multiple times
num_epochs = 3
for epoch in range(num_epochs):
    model.train() # train mode
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        # apply softmax function internall
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad() # to reset gradiesnt to zero
        # otherwise gradient will accumulate
        loss.backward() # calculate gradients in the CP
        optimizer.step() # Update the model parameters to minimze loss

        # LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    model.eval()

# since we do not have dropout and regularization
model.eval()
with torch.no_grad():
    outputs = model(X_train)

print(outputs)

torch.set_printoptions(sci_mode=False)
# probas = torch.softmax(outputs, dim=1)
# print(probas)

predictions = torch.argmax(outputs, dim=1)
print(predictions)

torch.sum(predictions == y_train)

def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions

        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

compute_accuracy(model, train_loader)
compute_accuracy(model, test_loader)



Epoch: 001/003 | Batch 000/003 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/003 | Train/Val Loss: 0.65
Epoch: 001/003 | Batch 002/003 | Train/Val Loss: 0.42
Epoch: 002/003 | Batch 000/003 | Train/Val Loss: 0.05
Epoch: 002/003 | Batch 001/003 | Train/Val Loss: 0.13
Epoch: 002/003 | Batch 002/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 000/003 | Train/Val Loss: 0.01
Epoch: 003/003 | Batch 001/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 002/003 | Train/Val Loss: 0.02
tensor([[ 2.9320, -4.2563],
        [ 2.6045, -3.8389],
        [ 2.1484, -3.2514],
        [-2.1461,  2.1496],
        [-2.5004,  2.5210]])
tensor([0, 0, 0, 1, 1])


1.0

#### 11a. A Simple Neural Net (regression variant)

Same DataLoader + training-loop pattern as above, applied to a regression task instead of
classification: a minimal linear model (y = 2x+1) extended into a real 2-hidden-layer MLP.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# simple linear model for y = 2x + 1
class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(1, 1)  # no hidden layer, no ReLU

    def forward(self, x):
        return self.lin(x) # linear Map. X()

model = SimpleLinear()
optimizer = optim.SGD(model.parameters(), lr=1e-2)  # small LR
loss_fn = nn.MSELoss()

# data
x_train = torch.randn(100, 1) * 10.0
y_train = 2.0 * x_train + 1.0 + torch.randn_like(x_train) * 0.5

# training
for epoch in range(200):
    optimizer.zero_grad() # zero accumulated gradient.
    pred = model(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss={loss.item():.4f}")

# quick check
x_test = torch.tensor([[5.0]])
pred = model(x_test).item()
print(f"Input=5, Pred={pred:.3f}, True={2*5+1}")

Epoch 0: Loss=398.6652
Epoch 20: Loss=85.7614
Epoch 40: Loss=18.6274
Epoch 60: Loss=4.2167
Epoch 80: Loss=1.1203
Epoch 100: Loss=0.4534
Epoch 120: Loss=0.3092
Epoch 140: Loss=0.2777
Epoch 160: Loss=0.2707
Epoch 180: Loss=0.2691
Input=5, Pred=11.026, True=11


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

torch.manual_seed(42)
device = ("mps" if torch.mps.is_available() else "cpu")

# MLP with 2 hidden layers and ReLU activation
class MLP(nn.Module):
    def __init__(self, in_dim=1, h1=64, h2=32, out_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_dim)

        # simple initialization (helps stable training)
        nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity='relu')
        nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity='relu')
        nn.init.uniform_(self.fc3.weight, -0.1, 0.1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x
    
# generate synthetic linear-ish data with noise (same ground truth: y=2x+1)
N = 2000
x = torch.randn(N, 1) * 10.0
y = 2.0 * x + 1.0 + torch.rand_like(x) * 0.5

# dataset + dataloader for mini-batch SGD
batch_size = 64
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# training : mini-batch gradient descent approximates full gradient using batches
epochs = 100
for epoch in range(epochs):
    epoch_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad() # clear gradients from previous batch
        yhat = model(xb) # forward compute predictions
        loss = loss_fn(yhat, yb) # compute batch loss
        loss.backward() # backprop: compute gradienst via autograd
        optimizer.step() # update params 
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= N
    if epoch % 10 == 0:
        print(f"Epoch {epoch: 03d}, Loss={epoch_loss:.4f}, Device={device}")

# evaluation on a single example
x_test = torch.tensor([[5.0]]).to(device)
with torch.no_grad():
    pred = model(x_test).cpu().item()
print(f"Input=5, Pred={pred:.3f}, true={2*5+1}")


Epoch  00, Loss=242.8731, Device=mps
Epoch  10, Loss=0.0248, Device=mps
Epoch  20, Loss=0.0233, Device=mps
Epoch  30, Loss=0.0221, Device=mps
Epoch  40, Loss=0.0238, Device=mps
Epoch  50, Loss=0.0234, Device=mps
Epoch  60, Loss=0.0238, Device=mps
Epoch  70, Loss=0.0227, Device=mps
Epoch  80, Loss=0.0228, Device=mps
Epoch  90, Loss=0.0227, Device=mps
Input=5, Pred=11.260, true=11


#### 12. PyTorch Modules & Containers

In [ ]:
import torch
import torch.nn as nn

class MyModule(nn.Module):
    def __init__(self, num_layers, input_dim, output_dim):
        super().__init__()
        # Module holds a list of layers, each is a linear layer
        self.layers = nn.ModuleList(
            [nn.Linear(input_dim, output_dim) for _ in range(num_layers)]
        )
    def forward(self, x):
        # Iterate through each layer in ModuleList
        for layer in self.layers:
            x = layer(x)
        return x

# ModuleList register each layer as a submoudle,
# so their parameters are included in model.parameters() 

In [2]:
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 30)
)

# Sequential - to define a model as a sequence of layers.
# It is a subclass of nn.Module that allows you to stack layers in a sequential manner.
print(model)

Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=30, bias=True)
)


In [3]:
class DictModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ModuleDict holds named layers
        self.layers = nn.ModuleDict(
            {
                "fc1": nn.Linear(10, 20),
                'fc2': nn.Linear(20, 5)
            }
        )
    def forward(self, x):
        x = self.layers['fc1'](x)
        x = self.layers['fc2'](x)
        return x

# ModuleDict is useful when you want to access layers by name
model = DictModel()
print(model)

DictModel(
  (layers): ModuleDict(
    (fc1): Linear(in_features=10, out_features=20, bias=True)
    (fc2): Linear(in_features=20, out_features=5, bias=True)
  )
)


#### 13. Saving and loading models

In [ ]:
# state_dict is a Python dictionary object that maps
# each layer in the model to its trainable parameters (weights & biases)

# torch.save(model.state_dict(), "model.pth")

# model = NeuralNetwork(2, 2) # needs to match orginaal model exactly
# model.load_state_dict(torch.load("model.pth", weights_only=True))
